In [1]:
import albumentations as A
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
import os
import random
from collections import defaultdict
from glob import glob

d:\LabsMIET\MLlab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 9999
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

In [3]:
class_to_idx = { "Апельсин": 0,
                 "Бананы": 1,
                 "Груши": 2, 
                 "Кабачки": 3, 
                 "Капуста": 4, 
                 "Картофель": 5, 
                 "Киви": 6, 
                 "Лимон": 7, 
                 "Лук": 8, 
                 "Мандарины": 9, 
                 "Морковь": 10, 
                 "Огурцы": 11, 
                 "Томаты": 12, 
                 "Яблоки зелёные": 13, 
                 "Яблоки красные": 14 }

In [4]:
class MyDataset(Dataset):
    def __init__(self, images_filepaths, name2label, transform=None):
        self.images_filepaths = images_filepaths
        self.transform = transform
        self.name2label = name2label

    def __len__(self):
        return len(self.images_filepaths)

    def __getitem__(self, idx):
        image_filepath = self.images_filepaths[idx]
        image = cv2.imdecode(np.fromfile(image_filepath, dtype=np.uint8), cv2.IMREAD_UNCHANGED)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = self.name2label[os.path.normpath(image_filepath).split(os.sep)[-3]]
        if self.transform is not None:
            image = self.transform(image=image)['image']
        return image, label


def train_test_split_from_directory(root_path, folder2class, train_size=0.8):
    train, test = [], []

    for class_name in os.listdir(root_path):
        class_path = os.path.join(root_path, class_name)
        if not os.path.isdir(class_path):
            continue

        for subclass_name in os.listdir(class_path):
            subclass_path = os.path.join(class_path, subclass_name)
            if not os.path.isdir(subclass_path):
                continue

            images = glob(os.path.join(subclass_path, '*.jpg')) + \
                     glob(os.path.join(subclass_path, '*.png')) + \
                     glob(os.path.join(subclass_path, '*.jpeg'))
            
            if len(images) == 0:
                continue
            
            # делим подклассы в пропорции 80/20
            random.shuffle(images)
            split_idx = int(train_size * len(images))

            if split_idx == 0 and len(images) > 0:
                split_idx = 1

            train.extend(images[:split_idx])
            test.extend(images[split_idx:])

    random.shuffle(train)
    random.shuffle(test)

    return train, test

Настройка датасета, writer для вывода, device - на чем обучается

In [5]:
from torch.utils.tensorboard import SummaryWriter

dataset_path = 'train/train'
train, test = train_test_split_from_directory(dataset_path, class_to_idx)

writer = SummaryWriter("kirillLogs")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
import torch.nn.functional as F

class FocalSmoothingLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.1, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.reduction = reduction

    def forward(self, logits, targets):
        n_classes = logits.size(-1)
        
        # Label smoothing  
        with torch.no_grad():
            true_dist = torch.full_like(logits, self.label_smoothing / (n_classes - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smoothing)
        
        # Log softmax 
        log_probs = F.log_softmax(logits, dim=-1)
        
        # Cross entropy с label smoothing
        ce_loss = -(true_dist * log_probs).sum(dim=-1)        
        
        # Focal-часть 
        pt = torch.exp(-ce_loss)                                 
        modulating_factor = (1 - pt) ** self.gamma
        
        #  Class weights 
        if self.alpha is not None:
            if self.alpha.dim() == 1:  
                alpha_t = self.alpha[targets]
            else:
                alpha_t = self.alpha
            loss = alpha_t * modulating_factor * ce_loss
        else:
            loss = modulating_factor * ce_loss
        
        # Редукция
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss

In [7]:
class HierarchicalSwinV2(nn.Module):
    def __init__(self, num_classes=15, model_name="swinv2_cr_tiny_224"):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=False,
            features_only=True  
        )

        feature_channels = self.backbone.feature_info.channels()

        self.heads = nn.ModuleList([
            nn.Sequential(  # stage 1 
                nn.AdaptiveAvgPool2d(1), 
                nn.Flatten(),
                nn.Linear(feature_channels[0], 96),
                nn.ReLU(),
            ),
            nn.Sequential(  # stage 2
                nn.AdaptiveAvgPool2d(1), 
                nn.Flatten(),
                nn.Linear(feature_channels[1], 128),
                nn.ReLU(),
            ),
            nn.Sequential(  # stage 3 
                nn.AdaptiveAvgPool2d(1), 
                nn.Flatten(),
                nn.Linear(feature_channels[2], 192),
                nn.ReLU(),
                nn.Dropout(0.15),
            ),
            nn.Sequential(  # stage 4 
                nn.AdaptiveAvgPool2d(1), 
                nn.Flatten(),
                nn.Linear(feature_channels[3], 256),
                nn.ReLU(),
                nn.Dropout(0.2),
            ),
        ])
        
        self.classifier = nn.Sequential(
            nn.Linear(672, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x) 
    
        pooled = []
        for i in range(4):
            feat = features[i]
            out = self.heads[i](feat)        
            pooled.append(out)
    
        fused = torch.cat(pooled, dim=1)     
    
        logits = self.classifier(fused)
        return logits

In [8]:
from collections import Counter
import os

def count_classes(image_paths, class_to_idx):
    counts = Counter()
    for path in image_paths:
        class_name = os.path.normpath(path).split(os.sep)[-3]
        class_idx = class_to_idx[class_name]
        counts[class_idx] += 1
    return counts

train_counts = count_classes(train, class_to_idx)
test_counts  = count_classes(test, class_to_idx)

In [9]:
from albumentations.pytorch import ToTensorV2

train_transforms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),

    A.Affine( # Масштаб, композиция, поворот
        shift_limit=0.1,
        scale_limit=0.15,
        rotate_limit=30,
        border_mode=0,          
        value=0,
        p=0.6
    ),

    A.ColorJitter( # Гамма
        brightness=0.25,
        contrast=0.25, 
        saturation=0.25, 
        hue=0.0, 
        p=0.5
    ),

    A.RandomShadow(p=0.25), # Тени
    A.RandomFog(p=0.12, fog_coef_lower=0.1, fog_coef_upper=0.4), # Туман
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.2), # Зернистость

    A.Normalize(mean=[0.485, 0.456, 0.406], std =[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transforms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

C:\Users\Kirill\AppData\Local\Temp\ipykernel_15144\1063698501.py:7: UserWarning: Argument(s) 'shift_limit, scale_limit, rotate_limit, value' are not valid for transform Affine
  A.Affine( # Масштаб, композиция, поворот
C:\Users\Kirill\AppData\Local\Temp\ipykernel_15144\1063698501.py:25: UserWarning: Argument(s) 'fog_coef_lower, fog_coef_upper' are not valid for transform RandomFog
  A.RandomFog(p=0.12, fog_coef_lower=0.1, fog_coef_upper=0.4), # Туман
C:\Users\Kirill\AppData\Local\Temp\ipykernel_15144\1063698501.py:26: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.2), # Зернистость


In [10]:
train_dataset = MyDataset(images_filepaths=train, name2label=class_to_idx, transform=train_transforms)
test_dataset = MyDataset(images_filepaths=test, name2label=class_to_idx, transform=val_transforms)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,  
    pin_memory=True,  
    persistent_workers=False  
)
test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0,  
    pin_memory=True,  
    persistent_workers=False
    )

In [11]:
import torch
from tqdm import tqdm

@torch.no_grad()
def evaluate(model, dataloader, loss_fn, device, desc="Val"):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    pbar = tqdm(dataloader, desc=desc, leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = loss_fn(logits, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        y_pred = logits.argmax(dim=1)
        total_correct += (y_pred == labels).sum().item()
        total_samples += batch_size

        avg_loss = total_loss / max(total_samples, 1)
        acc = total_correct / max(total_samples, 1)
        pbar.set_postfix(loss=f"{avg_loss:.4f}", acc=f"{acc:.4f}")

    avg_loss = total_loss / max(total_samples, 1)
    accuracy = total_correct / max(total_samples, 1)
    return accuracy, avg_loss


def train(model, criterion, optimizer, scheduler, train_loader, val_loader, device, writer=None, n_epochs=5):
    num_iter = 0

    for epoch in range(1, n_epochs + 1):
        model.train()

        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{n_epochs}", leave=True)

        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            
            optimizer.step()
            scheduler.step()

            # Накопим метрики для прогресс-бара
            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            y_pred = logits.argmax(dim=1)
            total_correct += (y_pred == labels).sum().item()

            avg_loss = total_loss / max(total_samples, 1)
            acc = total_correct / max(total_samples, 1)

            # tqdm live-metrics
            pbar.set_postfix(train_loss=f"{avg_loss:.4f}", train_acc=f"{acc:.4f}")

            # Логирование (по итерациям)
            num_iter += 1
            if writer is not None:
                writer.add_scalar("Loss/train", loss.item(), num_iter)
                writer.add_scalar("Accuracy/train", (y_pred == labels).float().mean().item(), num_iter)

        # Валидация (тоже с tqdm)
        val_acc, val_loss = evaluate(model, val_loader, criterion, device, desc=f"Val {epoch}/{n_epochs}")

        if writer is not None:
            writer.add_scalar("Loss/val", val_loss, num_iter)
            writer.add_scalar("Accuracy/val", val_acc, num_iter)

        print(f"Epoch {epoch}/{n_epochs}: val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

    return model

In [12]:
total_samples = sum(train_counts.values())         
num_classes = len(train_counts)

class_weights = torch.tensor(
    [total_samples / (num_classes * train_counts[i]) for i in range(num_classes)],
    dtype=torch.float32
)

class_weights = class_weights / class_weights.mean()

In [13]:
import torch.optim as optim

n_epochs = 30

model = HierarchicalSwinV2(num_classes=15).to(device)

criterion =  FocalSmoothingLoss(
    alpha=class_weights.to(device),
    gamma=2.0,
    label_smoothing=0.1
)

optimizer = optim.AdamW(model.parameters(), weight_decay=1e-2)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-3,                # пик lr
    epochs=n_epochs,
    steps_per_epoch=len(train_loader),
    pct_start=0.3,              # 30% времени на рост
    anneal_strategy='cos',
    div_factor=30.0,
    final_div_factor=1e4
)

In [14]:
model = train(
    model, 
    criterion, 
    optimizer, 
    scheduler,
    train_loader, 
    test_loader,  
    device, 
    writer, 
    n_epochs=n_epochs
)

Epoch 1/30: 100%|██████████| 976/976 [02:51<00:00,  5.69it/s, train_acc=0.2616, train_loss=1.5297]


Epoch 1/30: val_loss=1.1900  val_acc=0.4079


Epoch 2/30: 100%|██████████| 976/976 [02:47<00:00,  5.83it/s, train_acc=0.4025, train_loss=1.1917]


Epoch 2/30: val_loss=0.9625  val_acc=0.5104


Epoch 3/30: 100%|██████████| 976/976 [02:46<00:00,  5.87it/s, train_acc=0.4734, train_loss=1.0642]


...

Epoch 28/30: val_loss=0.3440  val_acc=0.8656


Epoch 29/30: 100%|██████████| 976/976 [03:08<00:00,  5.19it/s, train_acc=0.8661, train_loss=0.3348]


Epoch 29/30: val_loss=0.3403  val_acc=0.8656


Epoch 30/30: 100%|██████████| 976/976 [03:07<00:00,  5.20it/s, train_acc=0.8677, train_loss=0.3348]
                                                                                     

Epoch 30/30: val_loss=0.3395  val_acc=0.8671


In [15]:
import numpy as np
import torch
from sklearn.metrics import classification_report, confusion_matrix

@torch.no_grad()
def sklearn_report(model, dataloader, device, idx2class=None, digits=4):
    model.eval()

    y_true, y_pred = [], []

    for images, labels in dataloader:
        images = images.to(device, non_blocking=True)

        logits = model(images)
        preds = logits.argmax(dim=1).cpu().numpy()

        y_pred.append(preds)
        y_true.append(labels.numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    if idx2class is None:
        target_names = None
        labels = None
    else:
        labels = sorted(idx2class.keys())
        target_names = [idx2class[i] for i in labels]

    rep = classification_report(
        y_true, y_pred,
        labels=labels,
        target_names=target_names,
        digits=digits,
        zero_division=0
    )
    print(rep)

    if idx2class is not None:
        cm = confusion_matrix(y_true, y_pred, labels=labels)
        print("\nConfusion Matrix:")
        print(cm)

In [16]:
idx2class = {v: k for k, v in class_to_idx.items()}

sklearn_report(model, test_loader, device, idx2class=idx2class, digits=4)

                precision    recall  f1-score   support

      Апельсин     0.9467    0.9040    0.9249       177
        Бананы     0.8199    0.8148    0.8173       162
         Груши     0.7738    0.8025    0.7879        81
       Кабачки     0.6235    0.8983    0.7361        59
       Капуста     0.9745    0.9107    0.9415       168
     Картофель     0.9197    0.7975    0.8542       158
          Киви     0.5965    0.8947    0.7158        38
         Лимон     0.9375    0.9023    0.9195       133
           Лук     0.8647    0.8712    0.8679       132
     Мандарины     0.8743    0.9359    0.9040       156
       Морковь     0.8720    0.8790    0.8755       124
        Огурцы     0.9268    0.9744    0.9500       117
        Томаты     0.9032    0.9333    0.9180       150
Яблоки зелёные     0.8212    0.7294    0.7726       170
Яблоки красные     0.8489    0.8082    0.8281       146

      accuracy                         0.8671      1971
     macro avg     0.8469    0.8704    0.8542 